# vForge — 02. Fine-tune Gemma 4 E2B on Cloud TPU (LoRA, JAX/Keras 3.0)

Runs on **Colab → Runtime → Change runtime type → TPU v5e-1** (or any v5e).

For the Sprint we run on **Cloud TPU v5e-4** via the GCP credits.

## 1. Install JAX/Keras 3.0 + keras-hub

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'jax'
!pip -q install -U 'jax[tpu]' keras==3.6.0 keras-hub datasets huggingface_hub

## 2. Verify TPU

In [ ]:
import jax
print('JAX devices:', jax.devices())
import keras
print('Keras backend:', keras.backend.backend())

## 3. Load dataset

In [ ]:
from datasets import load_dataset
ds = load_dataset('json', data_files='https://huggingface.co/datasets/HuggingFaceH4/CodeAlpaca_20K/resolve/main/data/train-00000-of-00001-3a2050c1d3ebcacb.parquet', split='train[:1000]')
# Or use your own: ds = load_dataset('json', data_files='data/code.jsonl', split='train')
ds = ds.rename_columns({'prompt':'instruction','completion':'output'}) if 'prompt' in ds.column_names else ds
print(ds)

## 4. Format prompts

In [ ]:
def fmt(r):
    instr = r.get('instruction','').strip()
    inp = (r.get('input') or '').strip()
    out = (r.get('output') or '').strip()
    p = f'<start_of_turn>user\n{instr}'
    if inp: p += f'\n\n{inp}'
    p += f'<end_of_turn>\n<start_of_turn>model\n{out}<end_of_turn>'
    return p
texts = [fmt(r) for r in ds]
print(texts[0][:300])

## 5. Set up data-parallel TPU distribution

In [ ]:
devices = keras.distribution.list_devices()
print('devices:', devices)
if devices and any('tpu' in d.lower() for d in devices):
    keras.distribution.set_distribution(keras.distribution.DataParallel(devices=devices))

## 6. Load Gemma 4 E2B with LoRA

In [ ]:
import keras_hub, time
t0 = time.time()
causal = keras_hub.models.CausalLM.from_preset('gemma4_instruct_2b')
print(f'load: {time.time()-t0:.1f}s')
causal.backbone.enable_lora(rank=8)
causal.preprocessor.sequence_length = 512  # bump for production
causal.compile(
  optimizer=keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=0.01),
  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
  weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

## 7. Train

In [ ]:
import time, json
t0 = time.time()
history = causal.fit(x=texts, batch_size=4, epochs=1, verbose=2)
elapsed = time.time() - t0
metrics = {
  'hardware':'tpu', 'base_model':'google/gemma-4-E2B-it', 'rank':8,
  'epochs':1, 'batch_size':4, 'dataset_rows': len(texts),
  'train_time_sec': round(elapsed,2),
  'final_loss': float(history.history['loss'][-1]),
  'approx_train_tokens_per_sec': round(len(texts) * 512 / elapsed, 2),
}
print(json.dumps(metrics, indent=2))

## 8. Save adapter & report metrics

In [ ]:
from pathlib import Path
Path('out_tpu').mkdir(exist_ok=True)
causal.save_weights('out_tpu/lora.weights.h5')
Path('out_tpu/metrics.json').write_text(json.dumps(metrics, indent=2))
# POST these to your vForge backend:
# !curl -X POST $VFORGE/api/benchmarks/runs -H 'content-type: application/json' \
#   -d '{"name":"gemma-2-2b TPU","hardware":"tpu_v5e","benchmark_type":"training", "model":"google/gemma-4-E2B-it", "metrics": '"$(cat out_tpu/metrics.json)"'}'


## Next
- 03_finetune_gpu.ipynb — same data, GPU baseline.
- 04_benchmark_vllm.ipynb — inference throughput / latency / cost.
